## AgenticVision Main Orchestrator script:

In [4]:
# Version 1.0: Last updated 29/04/2026

# Change log:
# -
# -

# Notes:
# - Check file output window sizes - method to find optimal window size for model evaluation??
# - Update to include additional LLM API's. Need to intergrate Gemini and others. 3 total?

# Known issues:
# -
# -

In [1]:
#Load in libraries
import argparse
import json
import os
import signal
import subprocess
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional
from dotenv import load_dotenv

In [3]:
"""
capture_orchestrator.py
=======================

Orchestrates network traffic capture for LLM inference sessions and
records aligned metadata for downstream analysis.

This module:
- Launches `tshark` to capture packet-level traffic per session
- Executes prompts against configured LLM providers (e.g., Groq)
- Streams and records token-level response timing
- Saves outputs as:
    - `.pcap` files (raw network traffic)
    - `.json` metadata (session details)
    - `.tokens.json` (token-level timing traces)

Each session corresponds to a single (LLM, prompt) pair and is uniquely
identified for reproducibility and analysis.

Workflow:
    1. Start packet capture
    2. Send prompt to LLM API (streaming)
    3. Record token arrival times and full response
    4. Stop capture and persist artifacts

Outputs:
    captures/
        ├── <timestamp>_<llm>_<workload>_<id>.pcap
        ├── <timestamp>_<llm>_<workload>_<id>.json
        └── <timestamp>_<llm>_<workload>_<id>.tokens.json

Configuration:
    - LLM providers defined in `LLM_CONFIGS`
    - Prompts loaded dynamically from `prompt_bank`
    - API keys read from environment variables (.env supported)

Requirements:
    pip install groq openai anthropic httpx requests python-dotenv
    tshark installed and available on system path

    Linux:
        sudo apt install tshark

    macOS:
        brew install wireshark

Usage:
    python capture_orchestrator.py --runs 3 --delay 2

Arguments:
    --runs        Number of repetitions per (LLM, prompt) pair
    --delay       Delay (seconds) between sessions
    --llm         Optional substring filter for LLM names
    --workload    Optional filter for prompt workload type

Notes:
    - Requires sufficient privileges to capture network traffic
    - Ensure correct network interface is configured
    - Streaming responses are used to capture fine-grained latency

Limitations:
    - Token counts are approximated via whitespace splitting
    - No retry logic for failed API calls
    - Capture accuracy depends on system/network conditions
"""


# ── Optional SDK imports (only imported if key is set) ────────────────────────
#Import Groq LLM api
try:
    from groq import Groq
except ImportError:
    Groq = None  # type: ignore

#Import Gemini LLM api
try:
    from google import genai
except ImportError:
    genai = None  # type: ignore
 
# ─────────────────────────────────────────────────────────────────────────────
# Test toggle
# ─────────────────────────────────────────────────────────────────────────────

import importlib

#Set for full bank of prompts or test bank (smaller)
test_run = True

if test_run:
    import test_prompt_bank as prompt_bank
else:
    import prompt_bank

importlib.reload(prompt_bank) # To ensure using latest version of the prompt bank

PROMPT_BANK = prompt_bank.PROMPT_BANK
Prompt = prompt_bank.Prompt
 
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
#Setup data storage location
try:
    BASE_DIR = Path(__file__).resolve().parent.parent # for .py
except NameError:
    BASE_DIR = Path.cwd().parent  # fallback for notebooks

OUTPUT_DIR = BASE_DIR / "Datastore" / "raw_captures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
# Network interface for tshark. Use "any" on Linux, or find yours with:
#   tshark -D
CAPTURE_INTERFACE = r"\Device\NPF_{2C768E8E-C50A-4740-ABB3-2AF5ADB15BA2}"
 
# Seconds to wait between sessions to let TCP flows settle
INTER_SESSION_DELAY = 2.0
 
# Map of LLM providers. Each entry needs a callable that takes a prompt string
# and returns (response_text, latency_seconds).
# Add/remove providers here; the rest of the pipeline is automatic.

LLM_CONFIGS: Dict[str, Dict[str, Any]] = {
    "groq_qwen3_32b": {
        "api_key_env": "GROQ_API_KEY",
        "model": "qwen/qwen3-32b",
        "provider": "groq",
    },
    
    "groq_llama-3.3-70b-versatile": {
        "api_key_env": "GROQ_API_KEY",
        "model": "llama-3.3-70b-versatile",
        "provider": "groq",
    },
    
    "gemini_flash": {
        "api_key_env" :"GEMINI_API_KEY",
        "model": "gemini-2.5-flash",
        "provider": "gemini"
    },
    
}

 
# ─────────────────────────────────────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────────────────────────────────────
 
@dataclass
class SessionMeta:
    session_id: str
    llm_name: str
    model: str
    workload: str
    prompt_text: str
    expected_tokens: str
    tags: list
    timestamp_utc: str
    pcap_path: str
    response_text: str
    response_tokens_approx: int
    latency_seconds: float
    error: Optional[str]
 
 
# ─────────────────────────────────────────────────────────────────────────────
# LLM CALLERS
# ─────────────────────────────────────────────────────────────────────────────
    
 
def call_groq_stream(model: str, api_key: str, prompt: str):
    if Groq is None:
        raise ImportError("groq package not installed")

    client = Groq(api_key=api_key)

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    return stream


def call_gemini_stream(model: str, api_key: str, prompt: str):
    if genai is None:
        raise ImportError("google-genai package not installed")

    client = genai.Client(api_key=api_key)

    # Consume the stream HERE while client is still in scope
    for chunk in client.models.generate_content_stream(
        model=model,
        contents=[{"role": "user", "parts": [{"text": prompt}]}],
    ):
        yield chunk

 
CALLER_MAP = {
    "groq": call_groq_stream,
    "gemini": call_gemini_stream,

}

def extract_token_groq(chunk) -> Optional[str]:
    try:
        return chunk.choices[0].delta.content or None
    except (AttributeError, IndexError):
        return None

def extract_token_gemini(chunk) -> Optional[str]:
    try:
        return chunk.text or None
    except AttributeError:
        return None

TOKEN_EXTRACTOR_MAP = {
    "groq": extract_token_groq,
    "gemini": extract_token_gemini,
}
 
#Every provider needs both a caller and a token extractor
assert set(CALLER_MAP.keys()) == set(TOKEN_EXTRACTOR_MAP.keys())


# ─────────────────────────────────────────────────────────────────────────────
# TSHARK HELPERS
# ─────────────────────────────────────────────────────────────────────────────
 
def start_capture(pcap_path: Path) -> subprocess.Popen:
    cmd = [
        r"C:\Program Files\Wireshark\tshark.exe",
        "-i", CAPTURE_INTERFACE,
        "-w", str(pcap_path),
        "-q",
    ]

    print("tshark command:", " ".join(cmd))

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    return proc
 
 
def stop_capture(proc: subprocess.Popen) -> None:
    """Gracefully stop tshark and wait for it to flush the pcap."""
    proc.send_signal(signal.SIGTERM)
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SESSION RUNNER
# ─────────────────────────────────────────────────────────────────────────────
 
def run_session(llm_name: str, cfg: Dict[str, Any], prompt: Prompt) -> SessionMeta:
    session_id = str(uuid.uuid4())[:8]
    ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    safe_llm = llm_name.replace("/", "_")
    
    pcap_filename = f"{ts}_{safe_llm}_{prompt.workload}_{session_id}.pcap"
    pcap_path = (OUTPUT_DIR / pcap_filename).resolve()
    print("PCAP path:", pcap_path)
 
    api_key = os.environ.get(cfg["api_key_env"], "")
    if not api_key:
        print(f"  [SKIP] {llm_name}: env var {cfg['api_key_env']} not set")
        return SessionMeta(
            session_id=session_id,
            llm_name=llm_name,
            model=cfg["model"],
            workload=prompt.workload,
            prompt_text=prompt.text,
            expected_tokens=prompt.expected_tokens,
            tags=prompt.tags,
            timestamp_utc=ts,
            pcap_path=str(pcap_path),
            response_text="",
            response_tokens_approx=0,
            latency_seconds=0.0,
            error=f"API key not set: {cfg['api_key_env']}",
        )
 
    caller = CALLER_MAP.get(cfg["provider"])
    if caller is None:
        raise ValueError(f"Unknown provider: {cfg['provider']}")
 
    print(f"  [{llm_name}] Capturing -> {pcap_filename}")
    proc = start_capture(pcap_path)
    time.sleep(2) 
 
    response_text = ""
    latency = 0.0
    error = None
    
    try:
        stream = caller(cfg["model"], api_key, prompt.text)
        token_series = []
        full_text = ""

        extractor = TOKEN_EXTRACTOR_MAP.get(cfg["provider"])
        if extractor is None:
            raise ValueError(f"No token extractor for provider: {cfg['provider']}")

        t0 = time.perf_counter()

        for chunk in stream:
            token = extractor(chunk)
            if token:
                now = time.perf_counter()
                full_text += token
                token_series.append({"t_rel": now - t0, "token": token})

        latency = time.perf_counter() - t0
        response_text = full_text
        
        token_path = pcap_path.with_suffix(".tokens.json")
        with open(token_path, "w") as f:
            json.dump(token_series, f, indent=2)

    except Exception as exc:
        error = str(exc)
        print(f"API error: {exc}")
        
    finally:
        time.sleep(2) 
        stop_capture(proc)
 
    meta = SessionMeta(
        session_id=session_id,
        llm_name=llm_name,
        model=cfg["model"],
        workload=prompt.workload,
        prompt_text=prompt.text,
        expected_tokens=prompt.expected_tokens,
        tags=prompt.tags,
        timestamp_utc=ts,
        pcap_path=str(pcap_path),
        response_text=response_text,
        response_tokens_approx=len(response_text.split()),
        latency_seconds=round(latency, 4),
        error=error,
    )
 
    # Save metadata alongside the pcap
    
        
    meta_path = pcap_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(asdict(meta), f, indent=2)
 
    print(f"Latency={latency:.2f}s  ~{meta.response_tokens_approx} words")
    return meta
 
 
# ─────────────────────────────────────────────────────────────────────────────
# MAIN ORCHESTRATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
 
def run_all(runs: int = 1, delay: float = INTER_SESSION_DELAY,
            llm_filter: Optional[str] = None,
            workload_filter: Optional[str] = None) -> None:
    """
    Iterate over every (LLM x prompt) pair, repeated `runs` times.
    Optionally filter by llm_name or workload type.
    """
    targets = {k: v for k, v in LLM_CONFIGS.items()
               if llm_filter is None or llm_filter in k}
    prompts = [p for p in PROMPT_BANK
               if workload_filter is None or p.workload == workload_filter]
 
    total = len(targets) * len(prompts) * runs
    print(f"\nStarting capture run: {len(targets)} LLMs x {len(prompts)} prompts x {runs} run(s) = {total} sessions\n")
 
    completed = 0
    for run_idx in range(runs):
        print(f"── Run {run_idx + 1}/{runs} ─────────────────────────────────────────")
        for llm_name, cfg in targets.items():
            for prompt in prompts:
                run_session(llm_name, cfg, prompt)
                completed += 1
                print(f"  Progress: {completed}/{total}")
                time.sleep(delay)
 
    print(f"\nDone. Captures saved to: {OUTPUT_DIR.resolve()}")
 
 
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="LLM traffic capture orchestrator")
    parser.add_argument("--runs", type=int, default=1,
                        help="Number of repetitions per (LLM, prompt) pair")
    parser.add_argument("--delay", type=float, default=INTER_SESSION_DELAY,
                        help="Seconds between sessions")
    parser.add_argument("--llm", type=str, default=None,
                        help="Filter to LLM names containing this string")
    parser.add_argument("--workload", type=str, default=None,
                        help="Filter to a specific workload type")
    import sys

    if "ipykernel" in sys.modules:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()
 
    run_all(runs=args.runs, delay=args.delay, llm_filter=args.llm, workload_filter=args.workload)


Starting capture run: 3 LLMs x 10 prompts x 1 run(s) = 30 sessions

── Run 1/1 ─────────────────────────────────────────
PCAP path: E:\AS9Wa\MSc Thesis\AgenticVision\Datastore\raw_captures\20260506T102127Z_groq_qwen3_32b_short_factual_bcdb6a20.pcap
  [groq_qwen3_32b] Capturing -> 20260506T102127Z_groq_qwen3_32b_short_factual_bcdb6a20.pcap
tshark command: C:\Program Files\Wireshark\tshark.exe -i \Device\NPF_{2C768E8E-C50A-4740-ABB3-2AF5ADB15BA2} -w E:\AS9Wa\MSc Thesis\AgenticVision\Datastore\raw_captures\20260506T102127Z_groq_qwen3_32b_short_factual_bcdb6a20.pcap -q
Latency=0.87s  ~277 words
  Progress: 1/30
PCAP path: E:\AS9Wa\MSc Thesis\AgenticVision\Datastore\raw_captures\20260506T102134Z_groq_qwen3_32b_long_generation_6f5b8688.pcap
  [groq_qwen3_32b] Capturing -> 20260506T102134Z_groq_qwen3_32b_long_generation_6f5b8688.pcap
tshark command: C:\Program Files\Wireshark\tshark.exe -i \Device\NPF_{2C768E8E-C50A-4740-ABB3-2AF5ADB15BA2} -w E:\AS9Wa\MSc Thesis\AgenticVision\Datastore\raw_c